# Exploratory Data Analysis by Hypothesis
## Constraint-Aware Auto-Ordering for School Meal Subscriptions

This notebook loads the synthetic school-meal dataset and runs an EDA organised around the four research questions and hypotheses in the synopsis.

| RQ | Null hypothesis (H₀) | Alternative (H₁) |
|---|---|---|
| **H1** Repeat vs explore | A personalised, history-aware strategy has the same per-student Hit@1 as a popularity baseline | Personalised strategy has a higher Hit@1 |
| **H2** Menu change | Hit@1 is the same in the 10 school days after a menu-cycle change as mid-cycle | Hit@1 is lower after a change |
| **H3** Cold start | Personal-history predictions are no better than cohort predictions at any history length | Personal predictions overtake cohort predictions after *k* orders |
| **H4** Acceptance | Auto-order cancellation rate is the same whether or not the system's pick matches the student's preference | Cancellation is higher when the pick does not match |

The EDA computes **simple, transparent baselines** (popularity, repeat-favourite, cohort). These give preliminary evidence for each hypothesis, check that the data can support the tests, and set the bar the models must beat later. They are not the final models.

**How to run:** unzip the dataset into a folder, set `DATA_DIR` in the next cell, then *Run All*. It needs about 4 GB of RAM and takes a few minutes.

In [ ]:
# ---- configuration ----
DATA_DIR = "../data/raw"            # folder containing STUDENTLUNCH.csv etc.
TEST_START = "2025-04-01"      # temporal hold-out start (as in the synopsis)
SY2_START = "2024-08-01"       # students registered after this are 'new' (H3)
RANDOM_STATE = 42

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .3, "font.size": 10})
FIG_DIR = "figures"; os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG_DIR}/{name}.png", dpi=150); plt.show()

def wilson_ci(k, n, z=1.96):
    # 95% Wilson confidence interval for a proportion k/n
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = np.where(n > 0, k / np.maximum(n, 1), np.nan)
    den = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return p, centre - half, centre + half

def two_prop_test(k1, n1, k2, n2):
    # two-sided two-proportion z test
    p1, p2 = k1 / n1, k2 / n2; p = (k1 + k2) / (n1 + n2)
    se = np.sqrt(p * (1 - p) * (1 / n1 + 1 / n2)); z = (p1 - p2) / se
    return p1, p2, z, 2 * stats.norm.sf(abs(z))

## 1. Load the data
Raw files keep two quirks from the source system: a UTF-8 byte-order mark, and an empty-string export artifact (`""`). Both are cleaned on load.

In [ ]:
def load(name, **kw):
    df = pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv"), encoding="utf-8-sig", **kw)
    obj = df.select_dtypes("object").columns
    df[obj] = df[obj].replace('""', np.nan)          # export artifact -> missing
    return df

lunch = load("STUDENTLUNCH", dtype={"STUDENTID": "int32", "GRADEID": "int32", "MENUITEMID": "int16",
                                    "PORTIONCODE": "int8", "STUDENTTYPE": "category", "LUNCHTYPE": "category",
                                    "ORDERCHANNEL": "category", "ORDERCHANNELDETAIL": "category"},
             parse_dates=["DELIVERYDATE", "LUNCHCREATED", "LUNCHCANCELED"])
student  = load("STUDENT", parse_dates=["DATEREGISTERED"])
grade    = load("GRADE"); school = load("SCHOOL"); sgroup = load("SERVINGGROUP")
item     = load("MENUITEM"); cuisine = load("CUISINE")
item_alg = load("MENUITEMALLERGEN"); stu_alg = load("STUDENTALLERGENS"); allergen = load("ALLERGEN")
menu     = load("TBLPACKINGGROUPMENUS", usecols=["SCHOOLID", "PACKINGGROUPID", "DELIVERYDATE", "MENUITEMID", "PORTIONCODE"],
                parse_dates=["DELIVERYDATE"], dtype={"MENUITEMID": "int16", "PORTIONCODE": "int8"})
addon    = load("STUDENTLUNCHADDON", usecols=["STUDENTLUNCHID", "COMPONENTID", "ADDONCANCELED"])

for n, d in [("STUDENTLUNCH", lunch), ("STUDENT", student), ("MENU (portion rows)", menu), ("ADDON", addon),
             ("MENUITEM", item), ("SCHOOL", school), ("GRADE", grade)]:
    print(f"{n:22s} {d.shape[0]:>10,} rows  {d.shape[1]:>3} cols")

## 2. Data quality checks

In [ ]:
checks = {
    "Lunch date range": f"{lunch.DELIVERYDATE.min().date()} → {lunch.DELIVERYDATE.max().date()}",
    "ISSERVED distinct values (deprecated column)": lunch.ISSERVED.unique().tolist(),
    "Share of lunches cancelled": f"{lunch.LUNCHCANCELED.notna().mean():.1%}",
    "Missing GRADEID / MENUITEMID": int(lunch.GRADEID.isna().sum() + lunch.MENUITEMID.isna().sum()),
    "Lunch types": lunch.LUNCHTYPE.value_counts(normalize=True).round(3).to_dict(),
}
live_all = lunch[lunch.LUNCHCANCELED.isna()]
checks["Student-days with >1 live lunch (should be 0)"] = int(live_all.duplicated(["STUDENTID", "DELIVERYDATE"]).sum())
for k, v in checks.items(): print(f"{k:48s} {v}")
print("\nMissing values per column (STUDENTLUNCH):"); print(lunch.isna().mean().round(3).to_string())

## 3. Core preparation
* **Live lunches** are rows with an empty `LUNCHCANCELED`; `ISSERVED` is deprecated and ignored.
* **IOU** (emergency) lunches are excluded from modelling.
* Each lunch is linked to its **school, age band and packing group** (grade → serving group → packing group), so it can be matched to the menu published for that day.
* The **candidate set** for a prediction is that packing group's menu on that date, minus items containing the student's active allergens.

In [ ]:
band_map = {"PS": "K-2", "TK": "K-2", "K": "K-2", "1": "K-2", "2": "K-2", "3": "3-5", "4": "3-5", "5": "3-5",
            "6": "6-8", "7": "6-8", "8": "6-8", "9": "9-12", "10": "9-12", "11": "9-12", "12": "9-12",
            "Staff": "Staff", "Unknown": "Unknown"}
BANDS = ["K-2", "3-5", "6-8", "9-12", "Staff"]
g = grade.merge(sgroup[["SERVINGGROUPID", "PACKINGGROUPID"]], on="SERVINGGROUPID", how="left")
g["BAND"] = g.GRADE.astype(str).map(band_map)
lunch = lunch.merge(g[["GRADEID", "SCHOOLID", "PACKINGGROUPID", "GRADE", "BAND"]], on="GRADEID", how="left")
item = item.merge(cuisine, on="CUISINEID", how="left")
lunch = lunch.merge(item[["MENUITEMID", "ITEMNAME", "CUISINENAME", "ADVENTUROUSRATING"]], on="MENUITEMID", how="left")
lunch["WEEKDAY"] = lunch.DELIVERYDATE.dt.day_name().str[:3]
lunch["IS_SUB"] = lunch.STUDENTTYPE.astype(str).str.startswith("Sub")
lunch["CANCELLED"] = lunch.LUNCHCANCELED.notna()

live = lunch[~lunch.CANCELLED & (lunch.LUNCHTYPE != "IOU")].copy()
train = live[live.DELIVERYDATE < TEST_START]; test = live[live.DELIVERYDATE >= TEST_START]

# candidate sets: entree menu per packing group & day (one row per item)
cand = (menu[menu.PORTIONCODE == 1][["PACKINGGROUPID", "DELIVERYDATE", "MENUITEMID"]].drop_duplicates())
# allergen filter (true allergens; dietary preferences are not recoverable from item tables)
active = stu_alg[stu_alg.ACTIVECODE.astype(str).str.lower() == "true"][["STUDENTID", "ALLERGENID"]]
conflict = active.merge(item_alg[["MENUITEMID", "ALLERGENID"]], on="ALLERGENID")[["STUDENTID", "MENUITEMID"]].drop_duplicates()
conflict["CONFLICT"] = True

def candidates_for(events):
    # events: EVENT, STUDENTID, PACKINGGROUPID, DELIVERYDATE  ->  one row per eligible candidate item
    c = events.merge(cand, on=["PACKINGGROUPID", "DELIVERYDATE"], how="inner", suffixes=("_CHOSEN", ""))
    c = c.merge(conflict, on=["STUDENTID", "MENUITEMID"], how="left")
    return c[c.CONFLICT.isna()].drop(columns="CONFLICT")

print(f"Live lunches (excl. IOU): {len(live):,}   train: {len(train):,}   test: {len(test):,}")
print(f"Candidate rows (menu items per group-day): {len(cand):,}")

## 4. General EDA

In [ ]:
wk = live.set_index("DELIVERYDATE").resample("W").size()
fig, ax = plt.subplots(); ax.plot(wk.index, wk.values / 1000, color="#1f3b63")
ax.axvline(pd.Timestamp(TEST_START), ls="--", color="crimson", lw=1); ax.text(pd.Timestamp(TEST_START), ax.get_ylim()[1]*.9, " test window", color="crimson")
ax.set_title("Weekly live lunches"); ax.set_ylabel("thousands"); savefig("01_weekly_volume")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
live.WEEKDAY.value_counts().reindex(["Mon", "Tue", "Wed", "Thu", "Fri"]).plot.bar(ax=axes[0], color="#1f3b63", title="Lunches by weekday")
per_stu = live.groupby("STUDENTID").size()
axes[1].hist(per_stu.clip(upper=400), bins=40, color="#1f3b63"); axes[1].set_title("Live lunches per student (clipped at 400)")
live.BAND.value_counts().reindex(BANDS).plot.bar(ax=axes[2], color="#1f3b63", title="Lunches by age band")
savefig("02_distributions")

print("Students with ≥1 live lunch:", live.STUDENTID.nunique(), "of", len(student))
print("Share of live lunches from subscription plans:", f"{live.IS_SUB.mean():.1%}")
print("Students with no allergen record:", f"{(~student.STUDENTID.isin(stu_alg.STUDENTID)).mean():.1%}")

In [ ]:
# entree choices per school-day and subscription plan mix
per_day = cand.groupby(["PACKINGGROUPID", "DELIVERYDATE"]).size()
print("Entree choices per packing-group day:", per_day.describe()[["min", "mean", "max"]].round(1).to_dict())
plans = live[live.IS_SUB].STUDENTTYPE.astype(str).str.extract(r"Sub (\d)(\d)")
plans.columns = ["days_per_week", "sides_included"]
print("\nSubscription lunches by plan days/week:"); print(plans.days_per_week.value_counts(normalize=True).sort_index().round(3).to_string())

# sides attached to live lunches
sides = addon[addon.ADDONCANCELED.isna()].groupby("STUDENTLUNCHID").size()
k = live.STUDENTLUNCHID.map(sides).fillna(0).clip(upper=3)
print("\nSides per live lunch (0/1/2/3+):", k.value_counts(normalize=True).sort_index().round(3).to_dict())

In [ ]:
# age-related food preferences: share of each band's lunches going to the top items
top_items = live.ITEMNAME.value_counts().head(15).index
share = (pd.crosstab(live.ITEMNAME, live.BAND, normalize="columns").reindex(top_items)[BANDS] * 100)
fig, ax = plt.subplots(figsize=(8, 6)); im = ax.imshow(share.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(BANDS)), BANDS); ax.set_yticks(range(len(top_items)), top_items)
for i in range(share.shape[0]):
    for j in range(share.shape[1]): ax.text(j, i, f"{share.values[i, j]:.1f}", ha="center", va="center", fontsize=8)
ax.set_title("Share of lunches (%) by age band — top 15 items"); ax.grid(False); plt.colorbar(im); savefig("03_age_preferences")

cz = pd.crosstab(live.CUISINENAME, live.BAND, normalize="columns")[BANDS]
cz.T.plot.bar(stacked=True, figsize=(9, 4), colormap="tab20", title="Cuisine mix by age band").legend(bbox_to_anchor=(1, 1), fontsize=8)
savefig("04_cuisine_by_band")
chi2, p, dof, _ = stats.chi2_contingency(pd.crosstab(live.CUISINENAME, live.BAND)[BANDS])
print(f"Cuisine × age band: chi² = {chi2:,.0f}, dof = {dof}, p = {p:.2e}  → preferences differ by age")

## 5. H1 — Repeat versus explore
**Question:** do students have stable favourites, and how much does personalisation beat popularity?

The EDA measures (a) how repetitive each student is, (b) how this varies by age, and (c) the test-window Hit@1 of three transparent baselines:
* **Random:** expected Hit@1 = 1 / number of eligible items.
* **Popularity:** the most popular item at the student's school and age band in training data.
* **Repeat-favourite:** the item the student ordered most often in training data, among today's eligible items.

In [ ]:
s = live.sort_values(["STUDENTID", "DELIVERYDATE"])
s["REPEAT_PREV"] = s.MENUITEMID.eq(s.groupby("STUDENTID").MENUITEMID.shift())
stu = s.groupby("STUDENTID").agg(n=("MENUITEMID", "size"), uniq=("MENUITEMID", "nunique"),
                                 repeat_prev=("REPEAT_PREV", "mean"), band=("BAND", "last"))
stu["fav_share"] = s.groupby("STUDENTID").MENUITEMID.agg(lambda x: x.value_counts(normalize=True).iloc[0])
stu["variety_ratio"] = stu.uniq / stu.n
stu10 = stu[stu.n >= 10]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(stu10.variety_ratio, bins=40, color="#1f3b63"); axes[0].set_title("Variety ratio (unique items ÷ orders), students with ≥10 orders")
stu10.boxplot(column="fav_share", by="band", ax=axes[1], positions=None); axes[1].set_title("Share of orders on favourite item, by age band"); plt.suptitle("")
savefig("05_h1_repetition")
print(stu10.groupby("band")[["variety_ratio", "fav_share", "repeat_prev"]].median().reindex(BANDS).round(3))
kw = stats.kruskal(*[stu10[stu10.band == b].fav_share for b in BANDS if (stu10.band == b).any()])
print(f"\nKruskal-Wallis, favourite share across age bands: H = {kw.statistic:.1f}, p = {kw.pvalue:.2e}")

In [ ]:
# baselines on the test window
ev = test[["STUDENTLUNCHID", "STUDENTID", "PACKINGGROUPID", "SCHOOLID", "BAND", "DELIVERYDATE", "MENUITEMID"]].rename(
    columns={"STUDENTLUNCHID": "EVENT", "MENUITEMID": "CHOSEN"})
C = candidates_for(ev)
n_cand = C.groupby("EVENT").size()

pop = train.groupby(["SCHOOLID", "BAND", "MENUITEMID"]).size().rename("pop").reset_index()
glob = train.MENUITEMID.value_counts().rename("glob")
hist = train.groupby(["STUDENTID", "MENUITEMID"]).size().rename("hist").reset_index()
C = C.merge(pop, on=["SCHOOLID", "BAND", "MENUITEMID"], how="left").merge(hist, on=["STUDENTID", "MENUITEMID"], how="left")
C["glob"] = C.MENUITEMID.map(glob).fillna(0); C[["pop", "hist"]] = C[["pop", "hist"]].fillna(0)

def pick(df, cols):
    return df.sort_values(cols, ascending=False).drop_duplicates("EVENT").set_index("EVENT").MENUITEMID

res = ev.set_index("EVENT")[["STUDENTID", "BAND", "CHOSEN"]].copy()
res["n_cand"] = n_cand
res = res.dropna(subset=["n_cand"])
res["hit_random"] = 1 / res.n_cand
res["hit_pop"] = (pick(C, ["pop", "glob"]).reindex(res.index) == res.CHOSEN).astype(float)
res["hit_fav"] = (pick(C, ["hist", "pop", "glob"]).reindex(res.index) == res.CHOSEN).astype(float)

rows = []
for m in ["hit_random", "hit_pop", "hit_fav"]:
    k, n = res[m].sum(), len(res); p, lo, hi = wilson_ci(k, n); rows.append([m.replace("hit_", ""), float(p), float(lo), float(hi)])
h1 = pd.DataFrame(rows, columns=["baseline", "Hit@1", "CI low", "CI high"]).round(4); print(h1.to_string(index=False))
print("\nHit@1 by age band:"); print(res.groupby("BAND")[["hit_random", "hit_pop", "hit_fav"]].mean().reindex(BANDS).round(3))

In [ ]:
# per-student paired comparison (the H1 test design): repeat-favourite vs popularity
ps = res.groupby("STUDENTID")[["hit_pop", "hit_fav"]].mean()
ps = ps[res.groupby("STUDENTID").size() >= 5]
d = ps.hit_fav - ps.hit_pop
t, p = stats.ttest_rel(ps.hit_fav, ps.hit_pop); w = stats.wilcoxon(ps.hit_fav, ps.hit_pop)
dz = d.mean() / d.std(ddof=1)
print(f"Students with ≥5 test events: {len(ps):,}")
print(f"Mean per-student Hit@1: favourite {ps.hit_fav.mean():.3f} vs popularity {ps.hit_pop.mean():.3f}")
print(f"Paired t = {t:.2f}, p = {p:.2e};  Wilcoxon p = {w.pvalue:.2e};  Cohen's d_z = {dz:.2f}")
print("→ The synopsis powered H1 for d_z = 0.20 (n = 199). The observed baseline gap shows how much headroom personalisation has.")
plt.hist(d, bins=40, color="#1f3b63"); plt.axvline(0, color="k", lw=.8)
plt.title("Per-student Hit@1 difference: repeat-favourite − popularity"); savefig("06_h1_paired_difference")

## 6. H2 — Robustness to menu change
Menu cycles are **detected from the data** rather than assumed: for each school, the weekly line-up (item × weekday slots) is compared with the previous week. A spike in the change rate marks a new cycle. Test-window predictions within **10 school days** after a detected change are tagged *post-change*; the rest are *mid-cycle*.

In [ ]:
m1 = menu[menu.PORTIONCODE == 1][["SCHOOLID", "DELIVERYDATE", "MENUITEMID"]].drop_duplicates()
m1["WEEK"] = m1.DELIVERYDATE.dt.to_period("W").dt.start_time; m1["WD"] = m1.DELIVERYDATE.dt.dayofweek
m1["SLOT"] = m1.MENUITEMID.astype(str) + "_" + m1.WD.astype(str)
slots = m1.groupby(["SCHOOLID", "WEEK"]).SLOT.apply(set).reset_index().sort_values(["SCHOOLID", "WEEK"])
# compare each week with the union of the previous TWO open weeks (absorbs items that alternate A/B weeks
# and ignores holiday gaps); change = share of this week's slots not seen in those two weeks
slots["P1"] = slots.groupby("SCHOOLID").SLOT.shift(1); slots["P2"] = slots.groupby("SCHOOLID").SLOT.shift(2)
slots["NSLOTS"] = slots.SLOT.apply(len)
slots["CHANGE"] = [np.nan if not (isinstance(b, set) and isinstance(c, set)) else len(a - (b | c)) / len(a)
                   for a, b, c in zip(slots.SLOT, slots.P1, slots.P2)]
slots = slots[slots.NSLOTS >= 15]                      # skip short (holiday) weeks
weekly = slots.groupby("WEEK").CHANGE.mean()
fig, ax = plt.subplots(); ax.plot(weekly.index, weekly.values, color="#1f3b63")
ax.set_title("Share of this week's menu slots not offered in the previous two weeks (avg. across schools)")
savefig("07_h2_menu_change_signal")

thr = max(0.15, slots.CHANGE.quantile(.90))
chg = slots[slots.CHANGE > thr][["SCHOOLID", "WEEK"]]
print(f"Change threshold (90th percentile): {thr:.2f};  detected school-cycle changes: {len(chg):,}")
print("Most common change weeks:", chg.WEEK.value_counts().head(6).index.strftime("%Y-%m-%d").tolist())

In [ ]:
# tag each test event with school days since the most recent detected change at its school
sd = m1[["SCHOOLID", "DELIVERYDATE"]].drop_duplicates().sort_values(["SCHOOLID", "DELIVERYDATE"])
sd["DAYIDX"] = sd.groupby("SCHOOLID").cumcount()
cstart = chg.merge(sd, left_on="SCHOOLID", right_on="SCHOOLID")
cstart = cstart[cstart.DELIVERYDATE >= cstart.WEEK].sort_values("DELIVERYDATE").drop_duplicates(["SCHOOLID", "WEEK"])
cstart = cstart[["SCHOOLID", "DELIVERYDATE", "DAYIDX"]].rename(columns={"DELIVERYDATE": "CHG_DATE", "DAYIDX": "CHG_IDX"})
r2 = res.join(ev.set_index("EVENT")[["SCHOOLID", "DELIVERYDATE"]]).reset_index()
r2 = r2.merge(sd, on=["SCHOOLID", "DELIVERYDATE"], how="left").sort_values("DELIVERYDATE")
r2 = pd.merge_asof(r2, cstart.sort_values("CHG_DATE"), left_on="DELIVERYDATE", right_on="CHG_DATE", by="SCHOOLID", direction="backward")
r2["DAYS_SINCE"] = r2.DAYIDX - r2.CHG_IDX
r2["PHASE"] = np.where(r2.DAYS_SINCE.between(0, 9), "post-change", "mid-cycle")

tab = r2.groupby("PHASE").agg(events=("hit_fav", "size"), hit_fav=("hit_fav", "mean"), hit_pop=("hit_pop", "mean"))
print(tab.round(4))
a, b = r2[r2.PHASE == "mid-cycle"], r2[r2.PHASE == "post-change"]
p1, p2, z, p = two_prop_test(a.hit_fav.sum(), len(a), b.hit_fav.sum(), len(b))
print(f"\nRepeat-favourite Hit@1 mid-cycle {p1:.3f} vs post-change {p2:.3f}: z = {z:.2f}, p = {p:.2e}")
curve = r2[r2.DAYS_SINCE.between(0, 40)].groupby("DAYS_SINCE")[["hit_fav", "hit_pop"]].mean()
curve.plot(title="Hit@1 by school days since last menu change (test window)", color=["#1f3b63", "#c97b2a"]); savefig("08_h2_days_since_change")

In [ ]:
# what replaces a favourite that disappeared? (students whose training favourite is not on today's menu)
fav = train.groupby("STUDENTID").MENUITEMID.agg(lambda x: x.value_counts().index[0]).rename("FAV")
avail = C.merge(fav, left_on="STUDENTID", right_index=True)
avail = avail.assign(FAV_ON=avail.MENUITEMID == avail.FAV).groupby("EVENT").FAV_ON.any()
r2["FAV_AVAILABLE"] = r2.EVENT.map(avail).fillna(False).astype(bool)
print("Share of test events where the student's favourite is on the menu, by phase:")
print(r2.FAV_AVAILABLE.astype(float).groupby(r2.PHASE).mean().round(3))
gone = r2[~r2.FAV_AVAILABLE].merge(fav, left_on="STUDENTID", right_index=True)
gone = gone.merge(item[["MENUITEMID", "CUISINEID"]].rename(columns={"MENUITEMID": "FAV", "CUISINEID": "FAV_CUI"}), on="FAV")
gone = gone.merge(item[["MENUITEMID", "CUISINEID"]].rename(columns={"MENUITEMID": "CHOSEN"}), on="CHOSEN")
print(f"\nWhen the favourite is missing, the student picks the same cuisine {(gone.CUISINEID == gone.FAV_CUI).mean():.1%} of the time")

## 7. H3 — Cold start
For students who registered in the 2024–25 school year, each of their first 15 orders is scored with:
* **Cohort baseline:** the most popular eligible item for that school × age band in data *before* the order date (2023–24 history).
* **Personal favourite:** the item the student ordered most often in their *previous* orders (falls back to cohort when there is no history).

The point where the personal curve overtakes the cohort curve estimates the minimum history needed.

In [ ]:
new_ids = student[student.DATEREGISTERED >= SY2_START].STUDENTID
nl = live[live.STUDENTID.isin(new_ids)].sort_values(["STUDENTID", "DELIVERYDATE"]).copy()
nl["K"] = nl.groupby("STUDENTID").cumcount()            # number of prior orders
nl = nl[nl.K < 15]
print(f"New students: {len(new_ids):,};  with ≥10 live orders: {(live[live.STUDENTID.isin(new_ids)].groupby('STUDENTID').size() >= 10).sum():,}")

ev3 = nl[["STUDENTLUNCHID", "STUDENTID", "PACKINGGROUPID", "SCHOOLID", "BAND", "DELIVERYDATE", "MENUITEMID", "K"]].rename(
    columns={"STUDENTLUNCHID": "EVENT", "MENUITEMID": "CHOSEN"})
C3 = candidates_for(ev3)
coh = live[live.DELIVERYDATE < SY2_START].groupby(["SCHOOLID", "BAND", "MENUITEMID"]).size().rename("coh").reset_index()
C3 = C3.merge(coh, on=["SCHOOLID", "BAND", "MENUITEMID"], how="left"); C3["coh"] = C3.coh.fillna(0)
C3["glob"] = C3.MENUITEMID.map(glob).fillna(0)
# prior personal count of each candidate item, strictly before the order date (merge_asof)
h = nl[["STUDENTID", "MENUITEMID", "DELIVERYDATE"]].sort_values("DELIVERYDATE").copy()
h["PRIOR"] = h.groupby(["STUDENTID", "MENUITEMID"]).cumcount() + 1
C3 = pd.merge_asof(C3.sort_values("DELIVERYDATE"), h, on="DELIVERYDATE", by=["STUDENTID", "MENUITEMID"],
                   allow_exact_matches=False, direction="backward")
C3["PRIOR"] = C3.PRIOR.fillna(0)
e3 = ev3.set_index("EVENT")[["K", "BAND", "CHOSEN"]]
e3["hit_cohort"] = (pick(C3, ["coh", "glob"]).reindex(e3.index) == e3.CHOSEN).astype(float)
e3["hit_personal"] = (pick(C3, ["PRIOR", "coh", "glob"]).reindex(e3.index) == e3.CHOSEN).astype(float)
lc = e3.groupby("K").agg(n=("hit_cohort", "size"), cohort=("hit_cohort", "mean"), personal=("hit_personal", "mean"))
for col in ["cohort", "personal"]:
    _, lc[f"{col}_lo"], lc[f"{col}_hi"] = wilson_ci(lc[col] * lc.n, lc.n)
fig, ax = plt.subplots()
for col, c in [("cohort", "#c97b2a"), ("personal", "#1f3b63")]:
    ax.plot(lc.index, lc[col], marker="o", color=c, label=col); ax.fill_between(lc.index, lc[f"{col}_lo"], lc[f"{col}_hi"], color=c, alpha=.15)
ax.set_xlabel("Number of prior orders"); ax.set_ylabel("Hit@1"); ax.set_title("Cold-start learning curves (95% CI)"); ax.legend()
savefig("09_h3_learning_curves")
cross = lc[(lc.personal_lo > lc.cohort_hi)].index.min()
print(lc[["n", "cohort", "personal"]].round(3).T)
print(f"\nFirst history length where personal beats cohort with non-overlapping 95% CIs: {cross} prior orders")

In [ ]:
# does the crossover depend on age band?
byband = e3.groupby(["BAND", "K"])[["hit_cohort", "hit_personal"]].mean().reset_index()
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), sharey=True)
for ax, b in zip(axes, ["K-2", "3-5", "6-8", "9-12"]):
    x = byband[byband.BAND == b]; ax.plot(x.K, x.hit_cohort, color="#c97b2a"); ax.plot(x.K, x.hit_personal, color="#1f3b63"); ax.set_title(b)
axes[0].legend(["cohort", "personal"]); savefig("10_h3_by_band")

## 8. H4 — Auto-order acceptance
Legacy auto-ordered lunches are the only observable signal of parent acceptance. For each one, the system's pick is compared with a **preference proxy**: the student's most frequently chosen item among that day's eligible items, based on their own previous **manually placed** live lunches (before the delivery date). Cancellation rates are then compared for matches and mismatches.

In [ ]:
rates = lunch.groupby("LUNCHTYPE", observed=True).CANCELLED.agg(["sum", "size"])
rates["rate"], rates["lo"], rates["hi"] = wilson_ci(rates["sum"], rates["size"])
print("Cancellation rate by lunch type (95% CI):"); print(rates.round(4))
auto = lunch[lunch.LUNCHTYPE == "AutoOrder"].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
auto.groupby("WEEKDAY").CANCELLED.mean().reindex(["Mon", "Tue", "Wed", "Thu", "Fri"]).plot.bar(ax=axes[0], color="#1f3b63", title="Auto-order cancellation by weekday")
auto.groupby(auto.DELIVERYDATE.dt.to_period("M")).CANCELLED.mean().plot(ax=axes[1], color="#1f3b63", title="Auto-order cancellation by month")
savefig("11_h4_cancellation_patterns")

In [ ]:
ev4 = auto[["STUDENTLUNCHID", "STUDENTID", "PACKINGGROUPID", "DELIVERYDATE", "MENUITEMID", "CANCELLED", "BAND"]].rename(
    columns={"STUDENTLUNCHID": "EVENT", "MENUITEMID": "PICK"})
C4 = candidates_for(ev4)
manual = live[live.LUNCHTYPE.isin(["PreOrder", "LastMinute"]) & live.STUDENTID.isin(ev4.STUDENTID)]
h4 = manual[["STUDENTID", "MENUITEMID", "DELIVERYDATE"]].sort_values("DELIVERYDATE").copy()
h4["PRIOR"] = h4.groupby(["STUDENTID", "MENUITEMID"]).cumcount() + 1
C4 = pd.merge_asof(C4.sort_values("DELIVERYDATE"), h4, on="DELIVERYDATE", by=["STUDENTID", "MENUITEMID"],
                   allow_exact_matches=False, direction="backward")
C4["PRIOR"] = C4.PRIOR.fillna(0); C4["glob"] = C4.MENUITEMID.map(glob).fillna(0)
has_hist = C4.groupby("EVENT").PRIOR.sum() >= 5          # need some manual history to define a preference
pref = pick(C4, ["PRIOR", "glob"])
e4 = ev4.set_index("EVENT"); e4["PREF"] = pref; e4 = e4[e4.index.isin(has_hist[has_hist].index)]
e4["MATCH"] = e4.PICK == e4.PREF
t4 = e4.groupby("MATCH").CANCELLED.agg(["sum", "size"])
t4["rate"], t4["lo"], t4["hi"] = wilson_ci(t4["sum"], t4["size"])
print(t4.round(4))
p_mis, p_mat, z, p = two_prop_test(t4.loc[False, "sum"], t4.loc[False, "size"], t4.loc[True, "sum"], t4.loc[True, "size"])
odds = (p_mis / (1 - p_mis)) / (p_mat / (1 - p_mat))
print(f"\nCancellation: mismatch {p_mis:.3f} vs match {p_mat:.3f};  z = {z:.2f}, p = {p:.2e};  odds ratio = {odds:.2f}")
print(f"Share of auto-orders where the legacy pick matched the preference proxy: {e4.MATCH.mean():.1%}")
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(["match", "mismatch"], [t4.loc[True, "rate"], t4.loc[False, "rate"]], color=["#1f3b63", "#c97b2a"],
       yerr=[[t4.loc[True, "rate"] - t4.loc[True, "lo"], t4.loc[False, "rate"] - t4.loc[False, "lo"]],
             [t4.loc[True, "hi"] - t4.loc[True, "rate"], t4.loc[False, "hi"] - t4.loc[False, "rate"]]], capsize=6)
ax.set_title("Auto-order cancellation rate (95% CI)"); savefig("12_h4_match_vs_mismatch")

In [ ]:
# what happens after a cancelled auto-order? (same-day replacement = swap; none = skipped day)
canc = auto[auto.CANCELLED][["STUDENTID", "DELIVERYDATE", "MENUITEMID"]]
repl = canc.merge(live[["STUDENTID", "DELIVERYDATE", "MENUITEMID"]].rename(columns={"MENUITEMID": "REPLACED_WITH"}),
                  on=["STUDENTID", "DELIVERYDATE"], how="left")
print(f"Cancelled auto-orders replaced by another lunch the same day (swap): {repl.REPLACED_WITH.notna().mean():.1%}")
print("→ Swaps reveal the item the family actually wanted: useful labels for the RQ4 classifier.")

## 9. Sample-size feasibility check
Required sample sizes from the synopsis compared with what the data provides.

In [ ]:
avail = {
    "RQ1 (students with test-window lunches)": (199, test.STUDENTID.nunique()),
    "RQ2 (students with test-window lunches)": (675, test.STUDENTID.nunique()),
    "RQ3 (new students with ≥10 orders)": (1068, int((live[live.STUDENTID.isin(new_ids)].groupby("STUDENTID").size() >= 10).sum())),
    "RQ4 (students with ≥10 auto-ordered lunches)": (416, int((auto.groupby("STUDENTID").size() >= 10).sum())),
}
ss = pd.DataFrame(avail, index=["required", "available"]).T
ss["sufficient"] = ss.available >= ss.required
print(ss.to_string()); print(f"\nFinal N = max(required) = {ss.required.max():,} students → all RQs feasible: {ss.sufficient.all()}")

## 10. Summary of EDA findings

In [ ]:
summary = pd.DataFrame([
    ["H1", "Repeat-favourite vs popularity Hit@1 (test)", f"{res.hit_fav.mean():.3f} vs {res.hit_pop.mean():.3f} (random {res.hit_random.mean():.3f})"],
    ["H1", "Per-student paired effect size d_z", f"{dz:.2f}"],
    ["H2", "Repeat-favourite Hit@1 mid-cycle vs post-change", f"{p1:.3f} vs {p2:.3f}"],
    ["H3", "Personal beats cohort (non-overlapping CIs) from", f"{cross} prior orders"],
    ["H4", "Auto-order cancellation mismatch vs match", f"{p_mis:.3f} vs {p_mat:.3f} (OR {odds:.2f})"],
], columns=["Hypothesis", "Metric", "Result"])
print(summary.to_string(index=False))
print(f"\nFigures saved in ./{FIG_DIR}/")

### Notes for the report
* These are **baseline** results. They establish that each effect exists in the data and set the benchmark the LightGBM ranker (RQ1) and the other models must beat.
* All candidate sets respect the published menu and true allergens. Dietary preferences (beef, pork, vegetarian, vegan) cannot be recovered from the item tables, so they are not filtered here.
* The `_ground_truth` folder is not used. It is reserved for checking, at the end, whether the models recover the simulated behaviour.